In [1]:
# optimize cells based on kalmbach_2021
import neuron
from neuron import h
import matplotlib.pyplot as plt
from neuron.units import ms, mV
import numpy as np
pth_mod = '/Users/katharinaduecker/Documents/projects_brown/hnn-tuning/local_hnn/hnn_core/mod'
#figpth = '/Users/katharinaduecker/Documents/projects_brown/hnn_dev/figures/newl5_hay/'
# import os
# os.mkdir(figpth)
from cell_builder_utils import l5_params, _get_dends, _get_pyr_soma, Cell
from hay_mechs_utils import Mechs
from plot_utils import fig_conductances, fig_compartments
from functools import partial

from hnn_core.viz import plot_cell_morphology

import pandas as pd
from scipy import signal
from plot_utils import fig_conductances, fig_compartments
from matplotlib.gridspec import GridSpec
import os
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
from scipy.signal import find_peaks
from scipy.fft import fft, fftfreq
from scipy.signal import windows
from scipy.signal import chirp

import pygad
os.chdir('/Users/katharinaduecker/Documents/projects_brown/hnn-tuning')

from hay_mechs_utils import Mechs, Ih_condu_apical, linear_gbar


In [3]:
# load electrophysio from Kalmbach, 2021

df = pd.read_csv(os.path.join('data','kalmbach_2021_l5_electrophys.csv'))

df['etype'].value_counts() # 1 seems to be the E-type cells

l5df = df[df['etype'] == 1]

ephys_measures = dict()
ephys_measures['target_rmp'] = np.mean(l5df['RMP'].values)

ephys_measures['sag_ratio'] = np.mean(l5df['Sag_ratio'])

ephys_measures['rebound'] = np.mean(l5df['Rebound'])

ephys_measures['res_freq'] = np.mean(l5df['Fr'])

ephys_measures['ap_thresh'] = np.mean(l5df['AP_threshold'])

ephys_measures['ap_peak'] = np.mean(l5df['AP_peak'])

ephys_measures['fi_slope'] = np.mean(l5df['FI_slope_max'])

ephys_measures['ss_inp_res'] = np.mean(l5df['SSRn'])



In [4]:
# hand-tuned parameters
start_params = {'soma_gbar_NaTs2_t': 20_400e-4, 
                        'soma_gbar_SKv3_1': 6_930e-4,
                        'soma_gbar_Nap_Et2': 17.2e-4,
                        'soma_gbar_Ca_HVA': 9.92e-04*.75,
                        'soma_gbar_Ca_LVAst': 34.3e-4*.75,
                        'soma_gbar_SK_E2': 441e-4,
                        'soma_gbar_Ih': 1e-4*1.1,
                        'soma_gbar_Im': 0,
                        'soma_gbar_K_Pst': 22.3e-4,
                        'soma_gbar_K_Tst': 812e-4,
                        'soma_g_pas':  .338e-4,
                        'soma_e_pas': -90,
                        'soma_decay_CaDynamics_E2': 460,
                        'soma_gamma_CaDynamics_E2': .000501,
                        'basal_gbar_NaTs2_t': 20_400e-4*.1,
                        'basal_gbar_SKv3_1': 6_930e-4*0.1,
                        'basal_gbar_Ih': 5.14e-5,
                        'basal_g_pas': 1.75e-5,
                        'basal_e_pas':-90,
                        'apical_gbar_NaTa_t': 0.0213,
                        'apical_gbar_SKv3_1': 0.000261,
                        'apical_gbar_Ca_HVA': 2.78e-5,
                        'apical_gbar_Ca_LVAst': 93.5e-6,
                        'apical_gbar_SK_E2': 0.0012,
                        'apical_gbar_Ih': 1e-4,
                        'apical_gbar_Im': 0.0000675,
                        'apical_gbar_K_Pst': 0,
                        'apical_gbar_K_Tst': 0,
                        'apical_g_pas': 0.0000589*1.5,
                        'apical_e_pas': -85,
                        'apical_decay_CaDynamics_E2': 122,
                        'apical_gamma_CaDynamics_E2': .000501}


In [5]:
# Find resting membrane potential
section_params, end_pts, cell_tree = l5_params()

# build sections
section_names = list(end_pts.keys())

v_init = {'soma': -65,
            'basal_1': -65,
            'basal_2': -65,
            'basal_3': -65,
            'apical_oblique': -65,
            'apical_trunk': -65,
            'apical_1': -65,
            'apical_2': -60,
            'apical_tuft': -54}

sections = _get_dends(section_params, 'L5Pyr', section_names, end_pts, v_init=v_init)
sections['soma'] = _get_pyr_soma(section_params, 'L5Pyr', end_pts['soma'], v_init=v_init['soma'])

mech_class = Mechs(pth_mod)

# build model
hotzone = [1500, 2000]

In [107]:
def add_condu_cell(l5pyr, keys, conductances):

    h.distance(sec = l5pyr._nrn_sections['soma'])

    for key, val in zip(keys, conductances):
        idx1 = key.find("_")
        idx2 = key.find("_", idx1+1)
        attr = key[idx1+1:]
        mech = key[idx2+1:]
        #print(attr +','+mech+','+str(val))

        if key[:idx1] == 'soma':
            l5pyr._nrn_sections[key[:idx1]].insert(mech)
            for seg in l5pyr._nrn_sections['soma']:
                setattr(seg, attr, val)
        elif key[:idx1] == 'basal':
            for section in basal_sections:
                l5pyr._nrn_sections[section].insert(mech)
                for seg in l5pyr._nrn_sections[section]:
                    setattr(seg, attr, val)
        elif key[:idx1] == 'apical':
            for section in apical_sections:
                l5pyr._nrn_sections[section].insert(mech)
                for seg in l5pyr._nrn_sections[section]:
                    if mech == 'Ca_HVA':
                        distance = h.distance(seg.x, sec=seg.sec)
                        gbar = linear_gbar(val, val*9,1500, distance)
                        if (distance >= hotzone[0]) and (distance < hotzone[1]):
                                gbar *= 3
                    elif mech == 'Ca_LVAst':
                        distance = h.distance(seg.x, sec=seg.sec)
                        gbar = linear_gbar(val, val*4,1500, distance)
                        if (distance >= hotzone[0]) and (distance < hotzone[1]):
                                gbar *= 3
                    elif mech == 'Ih':
                        distance = h.distance(seg.x, sec=seg.sec)
                        gbar = Ih_condu_apical(val, distance)
                    else:
                        gbar = val
                    setattr(seg, attr, gbar)
    
    return l5pyr

In [108]:
prefixes = [f"RMP target: {np.round(ephys_measures['target_rmp'])} model ", 
            f"grad target: .05: ", 
            f"imped target: {ephys_measures['ss_inp_res']} model: ", 
            f"rebound target: {ephys_measures['rebound']} model: ", 
            f"sag target: {ephys_measures['sag_ratio']} model: ", 
            f"ap thresh target: {ephys_measures['ap_thresh']} model: ", 
            f"ap peak target: {ephys_measures['ap_peak']} model: ", 
            f"FI slope target: {ephys_measures['fi_slope']} model: ", 
            f"res freq target: {ephys_measures['res_freq']} model: "
            ]

In [ ]:
population_size = 2
initial_population_2d = np.tile(initial_population, (population_size, 1))
gene_space = {'low': .1, 'high': 2}

In [ ]:
# Use the fitness function with PyGAD
ga_instance = pygad.GA(num_generations=2,
                       num_parents_mating=10,
                       fitness_func=fitness_function,
                       num_genes=len(start_params),
                       initial_population=initial_population_2d,
                       gene_space=gene_space)

ga_instance.run()

RMP target: -62.0 model -69.24334302578295
grad target: .05: 0.02901828499940251
imped target: 38.76153461538461 model: 22.0


/var/folders/7z/wfcp96210zb700801j0vcbmw0000gn/T/ipykernel_60517/3023495042.py:73: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  peak_heights[i] = pk[1]['peak_heights']


rebound target: -0.26187276923076924 model: -0.189505806734696
sag target: 1.2549276846923076 model: 1.0
ap thresh target: -41.471223076923074 model: -37.84364506294741
ap peak target: 47.07976153846154 model: 43.171814579363804
FI slope target: 0.05227268461538462 model: 0.06600000000000006
res freq target: 3.238551538461538 model: 6.517380510544723
RMP target: -62.0 model -69.24331181764325
grad target: .05: 0.20710753289638717
imped target: 38.76153461538461 model: 22.0
rebound target: -0.26187276923076924 model: -0.1890528684784017
sag target: 1.2549276846923076 model: 1.0
ap thresh target: -41.471223076923074 model: -37.84364506294741
ap peak target: 47.07976153846154 model: 43.171814579363804
FI slope target: 0.05227268461538462 model: 0.06600000000000006
res freq target: 3.238551538461538 model: 6.517380510544723
RMP target: -62.0 model -69.24331181764325
grad target: .05: 0.20710753289638717
imped target: 38.76153461538461 model: 22.0
rebound target: -0.26187276923076924 model:

In [ ]:
best_condu = ga_instance.best_solution()[0]
l5pyr= Cell(sections, cell_tree, cell_type='L5_pyramidal', gid=0)
l5pyr.build()

l5pyr = add_condu_cell(l5pyr, keys,  best_condu)

# RMP

t = h.Vector().record(h._ref_t)
soma_voltg = h.Vector().record(l5pyr._nrn_sections['soma'](0.5)._ref_v)

h.celsius = 37
h.finitialize(-60 * mV)
h.continuerun(200 * ms)
rmp = soma_voltg[-3]

plt.plot(t, soma_voltg)